<h1 style="text-align: center;">02 Data Communication</h1>
<hr>


## **10. Acoustic Communication**

<div class="alert alert-light">

  **Name:** (add name here)

  **Matriculation Number:** (add matriculation number here)

</div>


### 10.1 Two AI Agents Switch from Speech to Data Tones

<p style="text-align: center;"><img src="../../.admin/img/agent-audiocom.png" alt="Illustration of two AI agents communicating through sound." style="max-width: 50%; height: auto;"></p>

Watch the [GibberLink demonstration](https://www.youtube.com/watch?v=EtNagNezo8w), in which two voice agents switch from spoken English to data transmitted as tones.


The demonstration begins with two conversational AI agents speaking English. After establishing that both participants are AI agents, they switch to an acoustic data protocol that both have been configured to use. Their messages are encoded into tones, transmitted through sound, and decoded by the receiving software. This provides a practical starting point for exploring how digital information becomes a physical signal.

**GibberLink is a real, working demonstration.** The misleading interpretation is that two AIs spontaneously invented a secret language. The developers explicitly instructed both agents to switch to an existing acoustic protocol when they identified the other participant as an AI agent, as explained in the [original project documentation](https://github.com/PennyroyalTea/gibberlink#how-it-works).

The agents begin in spoken English and establish that both support the alternative communication mode. Their software then uses **ggwave** to encode messages into sounds and decode received sounds back into data, similar in principle to a modem. The encoding and decoding are supplied by an existing library; they are not a language invented during the conversation. See the [ggwave project](https://github.com/ggerganov/ggwave) for the underlying implementation.

**The sounds really carry messages**, and people can recover those messages with the appropriate software. The developers explain that the [ggwave web demo](https://waver.ggerganov.com/) can decode messages from the demonstration video. This is an example of AI agents using a communication tool provided by their developers; it provides no evidence of self-awareness or an independently invented AI language. The connection to this notebook is the shared protocol that makes the tones interpretable: both ends need compatible rules for encoding, transmitting, and decoding the data.

The activities across this notebook and the next three build a simple acoustic protocol, investigate the recording in `secret_message.wav`, and include paired exercises to encode and decode messages. These activities connect the audible signals to the waveforms, modulation schemes, and receiver operations that make communication possible. These ideas also underpin radio systems, including the use of OFDM in Wi-Fi, although an acoustic channel has different physical properties and constraints.

The exercises build on waveforms, the Fourier transform, modulation, noise, and correlation. Revisit the earlier notebooks in this chapter for a refresher before continuing.


### 10.2 Audio as a Communication System

Everyday speech provides a familiar example of a communication system: a speaker turns a message into sound, the sound travels through the air, and a listener interprets it using a shared language. The following table gives a simplified mapping between speech and the building blocks introduced earlier in this chapter.

| Speech | Communication system |
|---|---|
| Mouth and vocal tract | Modulator and transmitter |
| Air | Channel |
| Ear and auditory system | Receiver and signal processing |
| Language comprehension | Decoding and interpretation |
| Shared language | Agreed coding scheme |

Successful communication depends on both a usable signal and a shared way of interpreting it. Speaking more loudly cannot resolve a language barrier, just as increasing transmission power cannot compensate for incompatible encodings. Distance, background noise, and echoes can also make speech harder to understand, limiting where and how quickly people can communicate reliably.

An acoustic data link uses a loudspeaker and microphone in place of the human speaker and listener. Software can generate and detect signal patterns that people would find difficult to interpret by ear, but it must still cope with the room, the audio hardware, and the need for an agreed protocol.


### 10.3 Text Becomes Bits

Transmitting a text message requires representing its characters as numbers and then as bits. This notebook uses **ASCII**, which assigns values from 0 to 127 to 128 characters. Although those values require only seven bits, the implementation stores and transmits each character in an eight-bit byte, so the most significant bit is always zero.

UTF-8 preserves these one-byte representations for ASCII characters while also supporting characters outside the ASCII set through multibyte sequences. To keep the encoding step simple, the exercises accept only ASCII input. Type a message in the box below and click **Encode** to inspect its bit sequence and compare the character, byte, and bit counts.


In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, FloatRangeSlider, SelectionSlider, Text, Button, Output, HBox
from IPython.display import Audio, display

for _root in (Path.cwd(), *Path.cwd().parents):
    ACOUSTIC_SUPPORT = _root / ".admin" / "acoustic_workshop"
    if (ACOUSTIC_SUPPORT / "acoustic.py").is_file():
        break
else:
    raise FileNotFoundError("Start Jupyter from inside the MobileComputing repository.")

if str(ACOUSTIC_SUPPORT) not in sys.path:
    sys.path.insert(0, str(ACOUSTIC_SUPPORT))

import acoustic as ac

ACOUSTIC_ASSETS = ACOUSTIC_SUPPORT / "assets"

rate = ac.SAMPLE_RATE
WIDE = {"description_width": "initial"}    # so slider labels are readable

message_box = Text(value="a", description="Message", style=WIDE)
encode_button = Button(description="Encode", button_style="primary")
encode_output = Output()

def show_encoding(_=None):
    encode_output.clear_output()
    with encode_output:
        text = message_box.value
        try:
            data = text.encode("ascii")
        except UnicodeEncodeError:
            print("ASCII only. Remove the non-ASCII characters and try again.")
            return
        bits = ac.text_to_bits(text)
        print(f"text      {text!r}")
        print(f"bits      {''.join(str(b) for b in bits)}")
        print(f"\n{len(text)} characters -> {len(data)} bytes -> {len(bits)} bits")

encode_button.on_click(show_encoding)
display(HBox([message_box, encode_button]), encode_output)
show_encoding()


### 10.4 Audio Characteristics

A sinusoidal acoustic signal can be described mathematically in the same way as a sinusoidal electromagnetic signal, even though the physical waves are different. Its amplitude, frequency, and phase determine the waveform:

$$s(t) = A\,\sin\!\left(2\pi f t + \varphi\right)$$

| Parameter | Symbol | Meaning |
|---|---|---|
| Amplitude | $A$ | Peak deviation from zero |
| Frequency | $f$ | Number of oscillations per second, in Hz |
| Phase | $\varphi$ | Position within the cycle at $t = 0$ |

Use the controls below to vary each parameter independently and compare the plotted waveform with the one-second audio clip. Frequency is closely related to perceived pitch, while amplitude affects sound level when playback gain is fixed. Automatic audio normalization is disabled, so the amplitude slider changes both the waveform and playback sound level. Keep the device volume fixed when comparing amplitudes.

For an isolated, steady tone, changing the starting phase is generally not audible, even though the shift is visible in the plot. This differs from changing phase during a signal, as in phase modulation, where transitions can affect the waveform and its sound.


In [ ]:
tone_seconds = 1.0
plot_ms = 10                     # width of the drawn window

amp_slider = FloatSlider(min=0.1, max=1.0, step=0.05, value=0.5,
                         description="Amplitude A", continuous_update=False, style=WIDE)
freq_slider = FloatSlider(min=100, max=2000, step=20, value=440,
                          description="Frequency f (Hz)", continuous_update=False, style=WIDE)
phase_slider = FloatSlider(min=-180, max=180, step=15, value=0,
                           description="Phase phi (deg)", continuous_update=False, style=WIDE)

def make_tone(seconds):
    t = np.arange(int(seconds * rate)) / rate
    return amp_slider.value * np.sin(2 * np.pi * freq_slider.value * t
                                     + np.deg2rad(phase_slider.value))

def make_tone_figure():
    wave = make_tone(plot_ms / 1000)
    t_ms = np.arange(len(wave)) / rate * 1000
    figure, axis = plt.subplots(figsize=(11, 2.8))
    axis.plot(t_ms, wave, linewidth=1.0)
    axis.set_xlabel("Time (ms)")
    axis.set_ylabel("Amplitude")
    axis.set_ylim(-1.05, 1.05)
    axis.margins(x=0)
    axis.set_title(f"A = {amp_slider.value:.2f}   f = {freq_slider.value:.0f} Hz"
                   f"   phi = {phase_slider.value:.0f} deg")
    figure.tight_layout()
    return figure

display(HBox([amp_slider, freq_slider, phase_slider]))

tone_figure = make_tone_figure()
tone_handle = display(tone_figure, display_id=True)
plt.close(tone_figure)
tone_audio = display(Audio(make_tone(tone_seconds), rate=rate, normalize=False), display_id=True)

def update_tone(change=None):
    tone_figure = make_tone_figure()
    tone_handle.update(tone_figure)
    plt.close(tone_figure)
    tone_audio.update(Audio(make_tone(tone_seconds), rate=rate, normalize=False))

for slider in (amp_slider, freq_slider, phase_slider):
    slider.observe(update_tone, names="value")


### 10.5 Modulating a Bitstream: ASK, FSK, and PSK

The techniques introduced in [Digital Modulation](02.4_Digital_Modulation.ipynb) also apply to acoustic communication. To turn a bitstream into sound, the transmitter varies the amplitude, frequency, or phase of a carrier tone according to the data. The examples below use two possible signal states, with each state representing one bit.

| Varied parameter | Scheme | Mapping used here |
|---|---|---|
| Amplitude, $A$ | Amplitude-shift keying (ASK) | Silence for 0, a tone for 1 |
| Frequency, $f$ | Frequency-shift keying (FSK) | One frequency for 0, another for 1 |
| Phase, $\varphi$ | Phase-shift keying (PSK) | A reference phase for 0, a 180-degree shift for 1 |

The ASK example uses **on-off keying (OOK)**, the special case in which one state has zero amplitude. Each panel uses the same eight-bit sequence and symbol duration, making it easier to compare the three schemes. The plots and audio players both use only the first four symbols (bits `1011`).

Listen to each waveform and use the ASK amplitude and FSK frequency controls to explore how the signal changes. Compare the visible changes with the audible changes. Automatic audio normalization is disabled for these examples, so the ASK amplitude control also changes playback sound level.


In [ ]:
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0], dtype=np.uint8)
symbol_ms = 100                 # one symbol length, shared by all three schemes
carrier_hz = 400.0             # carrier for ASK and PSK
fsk_zero_hz = 400.0            # FSK tone for a 0, fixed
show_symbols = 4               # leading symbols plotted and played

samples_per_symbol = int(symbol_ms / 1000 * rate)
t_symbol = np.arange(samples_per_symbol) / rate
draw_len = show_symbols * samples_per_symbol
bit_string = "".join(str(b) for b in bits)

def keyed(tone_for_0, tone_for_1):
    return np.concatenate([tone_for_1 if b else tone_for_0 for b in bits])

def ask_wave():
    on = ask_amp.value * np.sin(2 * np.pi * carrier_hz * t_symbol)
    return keyed(np.zeros_like(t_symbol), on)

def fsk_wave():
    return keyed(np.sin(2 * np.pi * fsk_zero_hz * t_symbol),
                 np.sin(2 * np.pi * fsk_one_hz.value * t_symbol))

def psk_wave():
    carrier = np.sin(2 * np.pi * carrier_hz * t_symbol)
    return keyed(carrier, np.sin(2 * np.pi * carrier_hz * t_symbol + np.pi))

def make_panel(wave, title):
    segment = wave[:draw_len]
    figure, axis = plt.subplots(figsize=(11, 1.9))
    axis.plot(np.arange(len(segment)) / rate * 1000, segment, linewidth=0.8)
    axis.set_xlabel("Time (ms)")
    axis.set_ylabel("Amplitude")
    axis.set_ylim(-1.05, 1.05)
    axis.margins(x=0)
    axis.set_title(title)
    figure.tight_layout()
    return figure

ask_amp = FloatSlider(min=0.1, max=1.0, step=0.05, value=0.5,
                      description="ASK amplitude A", continuous_update=False, style=WIDE)
fsk_one_hz = FloatSlider(min=500, max=1200, step=20, value=800,
                         description="FSK 1-tone f1 (Hz)", continuous_update=False, style=WIDE)

def ask_title():
    return (f"ASK   A = {ask_amp.value:.2f}   {carrier_hz:.0f} Hz carrier"
            f"   {symbol_ms} ms/symbol")

def fsk_title():
    return (f"FSK   f0 = {fsk_zero_hz:.0f} Hz   f1 = {fsk_one_hz.value:.0f} Hz"
            f"   {symbol_ms} ms/symbol")

def psk_title():
    return (f"PSK   {carrier_hz:.0f} Hz carrier   180 deg flip for a 1"
            f"   {symbol_ms} ms/symbol")

display(HBox([ask_amp, fsk_one_hz]))
print(f"bits {bit_string}   plot shows the first {show_symbols} symbols"
      f"   audio plays the same {show_symbols} symbols")

ask_figure = make_panel(ask_wave(), ask_title())
ask_handle = display(ask_figure, display_id=True)
plt.close(ask_figure)
ask_audio = display(Audio(0.4 * ask_wave()[:draw_len], rate=rate, normalize=False), display_id=True)

fsk_figure = make_panel(fsk_wave(), fsk_title())
fsk_handle = display(fsk_figure, display_id=True)
plt.close(fsk_figure)
fsk_audio = display(Audio(0.4 * fsk_wave()[:draw_len], rate=rate, normalize=False), display_id=True)

psk_figure = make_panel(psk_wave(), psk_title())
psk_handle = display(psk_figure, display_id=True)
plt.close(psk_figure)
psk_audio = display(Audio(0.4 * psk_wave()[:draw_len], rate=rate, normalize=False), display_id=True)

def update_ask(change=None):
    figure = make_panel(ask_wave(), ask_title())
    ask_handle.update(figure)
    plt.close(figure)
    ask_audio.update(Audio(0.4 * ask_wave()[:draw_len], rate=rate, normalize=False))

def update_fsk(change=None):
    figure = make_panel(fsk_wave(), fsk_title())
    fsk_handle.update(figure)
    plt.close(figure)
    fsk_audio.update(Audio(0.4 * fsk_wave()[:draw_len], rate=rate, normalize=False))

ask_amp.observe(update_ask, names="value")
fsk_one_hz.observe(update_fsk, names="value")

### 10.6 Symbol Rate, Bit Rate, and What Limits Them

A **symbol** selects one of the available signal states for a duration $T_s$. The **symbol rate** $R_s$ specifies how many symbols are transmitted per second. If each symbol represents $\log_2 M$ bits, where $M$ is the number of available states, the corresponding bit rate is

$$R_s = \frac{1}{T_s}\ \ \text{[symbol/s, or baud]}, \qquad
  R_b = R_s \cdot \log_2 M\ \ \text{[bit/s]}.$$

All three examples above use $M = 2$, so each symbol carries one bit and $R_b = R_s$. Shortening the symbols therefore increases the bit rate, but it also leaves the receiver less time to distinguish the signal states. In a real link, the usable bandwidth, noise, and echoes constrain how far the rate can increase while maintaining reliable detection.

Use the slider below to shorten the symbols in the FSK signal and listen as the two tones become harder to distinguish as separate notes. The plot keeps the same time window so that more symbols visibly fit into it as the rate increases. Human perception of individual tone changes differs from a software receiver's ability to decode them, so this listening experiment does not establish the link's maximum bit rate. Also edit the message and the two frequencies in the code to explore other signals.


In [ ]:
link_bits = ac.text_to_bits("HELLO")
f0_hz, f1_hz = 400.0, 800.0   # binary FSK: 400 Hz for a 0, 800 Hz for a 1
window_ms = 250               # fixed time window, so shorter symbols visibly pack in

link_slider = IntSlider(min=1, max=120, step=1, value=40,
                        description="Symbol length (ms)",
                        continuous_update=False, style=WIDE)

def link_wave(symbol_ms):
    n = int(round(rate * symbol_ms / 1000))
    freq = np.repeat(np.where(link_bits == 1, f1_hz, f0_hz), n)   # frequency held per symbol
    phase = np.cumsum(2 * np.pi * freq / rate)                    # continuous phase, no clicks
    return np.sin(phase)

def make_link_figure(symbol_ms):
    shown = link_wave(symbol_ms)[: int(round(window_ms / 1000 * rate))]
    figure, axis = plt.subplots(figsize=(11, 2.2))
    axis.plot(np.arange(len(shown)) / rate * 1000, shown, linewidth=0.8)
    if window_ms / symbol_ms <= 16:
        for k in range(int(window_ms // symbol_ms) + 1):
            axis.axvline(k * symbol_ms, color="0.85", linewidth=0.6, zorder=0)
    axis.set_xlim(0, window_ms)
    axis.set_ylim(-1.05, 1.05)
    axis.set_xlabel("Time (ms)")
    axis.set_ylabel("Amplitude")
    axis.set_title(f"HELLO as binary FSK   {symbol_ms} ms/symbol"
                   f"   {1000 / symbol_ms:.0f} bit/s   ({window_ms} ms window)")
    figure.tight_layout()
    return figure

display(link_slider)

link_figure = make_link_figure(link_slider.value)
link_handle = display(link_figure, display_id=True)
plt.close(link_figure)
link_audio = display(Audio(link_wave(link_slider.value), rate=rate), display_id=True)

def update_link(change=None):
    figure = make_link_figure(link_slider.value)
    link_handle.update(figure)
    plt.close(figure)
    link_audio.update(Audio(link_wave(link_slider.value), rate=rate))

link_slider.observe(update_link, names="value")

### 10.7 Secret Message

The recording in `secret_message.wav` connects the next three notebooks through the task of recovering its message. Begin by listening to it and inspecting its waveform and spectrum, using the range slider to zoom into different sections. Both graphs use the same selected time interval: the upper graph shows its waveform, and the lower graph shows its frequency spectrum. Look for changes in the signal's structure and consider what role each section might play in helping a receiver detect and decode the transmission (as requested below, in the exercise).


In [ ]:
secret, secret_rate = ac.load_wav(ACOUSTIC_ASSETS / "secret_message.wav")
secret_duration = len(secret) / secret_rate
print(f"{secret_duration:.2f} s at {secret_rate} Hz")
display(Audio(secret, rate=secret_rate))

def make_secret_figure(window):
    first = max(0, int(round(window[0] * secret_rate)))
    last = min(len(secret), int(round(window[1] * secret_rate)))
    piece = secret[first:last]
    time = np.arange(first, last) / secret_rate
    figure, (wave_axis, spectrum_axis) = plt.subplots(2, 1, figsize=(11, 6.0))

    wave_axis.plot(time, piece, linewidth=0.4, color="tab:blue")
    wave_axis.set_xlabel("Time (s)")
    wave_axis.set_ylabel("Amplitude")
    wave_axis.set_title("Waveform of the selected window")
    if window[1] > window[0]:
        wave_axis.set_xlim(window)
    wave_axis.margins(x=0)

    if len(piece) >= 16:
        frequencies, magnitude = ac.spectrum(piece, secret_rate)
        band = frequencies <= 3600
        spectrum_axis.plot(frequencies[band], magnitude[band],
                           linewidth=0.8, color="tab:blue")
    else:
        spectrum_axis.text(0.5, 0.5, "Select a wider time window to show the spectrum.",
                           transform=spectrum_axis.transAxes, ha="center", va="center")
    spectrum_axis.set_xlim(0, 3600)
    spectrum_axis.set_xlabel("Frequency (Hz)")
    spectrum_axis.set_ylabel("Magnitude")
    spectrum_axis.set_title("Spectrum of the selected window")

    figure.suptitle(f"secret_message.wav   {window[0]:.2f} s to {window[1]:.2f} s")
    figure.tight_layout()
    return figure

secret_slider = FloatRangeSlider(min=0.0, max=round(secret_duration, 2), step=0.05,
                                 value=[0.0, round(secret_duration, 2)], readout_format=".2f",
                                 description="Window (s)", continuous_update=False, style=WIDE)
display(secret_slider)

secret_figure = make_secret_figure(secret_slider.value)
secret_handle = display(secret_figure, display_id=True)
plt.close(secret_figure)

def update_secret_plot(change=None):
    secret_figure = make_secret_figure(secret_slider.value)
    secret_handle.update(secret_figure)
    plt.close(secret_figure)

secret_slider.observe(update_secret_plot, names="value")



---

### 10.8 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in the responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Which of the three modulation schemes explored here (ASK, FSK, or PSK) would be suitable for an acoustic link? Explain the choice in terms of the channel conditions and the assumed receiver requirements.**

(answer here)


**Listen again to the PSK signal and describe the audible signal. Are any changes associated with the modulation distinguishable, and how does this compare with changing the starting phase of the isolated tone in Section 10.4?**

(answer here)


**Using the symbol-duration slider in Section 10.6, shorten the symbols until the two FSK tones are no longer audible as separate notes. Report the symbol duration and corresponding bit rate, and explain why this perceptual limit does not necessarily prevent a software receiver from decoding the signal.**

(answer here)


**Suppose the bit rate of an acoustic link needs to double in a large lecture hall without shortening the symbols. Describe one way to achieve this and explain the tradeoff it introduces.**

(answer here)


**Audio systems commonly use sampling rates such as 44.1 kHz or 48 kHz. How do these rates relate to the range of audible frequencies and the sampling theorem? Explain how lowering or raising the sampling rate would affect the frequencies that can be represented and the amount of sample data generated per second.**

(answer here)


**Use the window slider in Section 10.7 to identify the three main sections of `secret_message.wav`. Report their approximate start and end times and suggest what purpose each section might serve.**

(answer here)


**OPTIONAL: Have all tasks in this notebook been read and understood? Are there any shortcomings, missing parts, or sections that should be improved?**

Please help the teaching team improve the notebook by identifying any unclear tasks or explanations and suggesting improvements.

(answer here)
